# Course 4 — Naive Bayes

Conditional independence within each class: a surprisingly powerful shortcut that
scales to millions of features and handles mixed data types naturally.

**Sections**
1. The Naive Bayes assumption — from LDA to conditional independence (0:00–0:20)
2. Three choices for the marginal density (0:20–0:40)
3. Toy example: NB vs LDA decision regions (0:40–1:00)
4. Connections: NB ↔ LDA/QDA, NB ↔ GAMs (1:00–1:15)
5. sklearn variants: GaussianNB, MultinomialNB, text classification (1:15–1:30)
6. Benchmark: when NB wins and loses (1:30–1:45)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import LabelEncoder

default = load_dataset('default')
iris    = load_dataset('iris')
d = default.copy()
d['y'] = (d['default'] == 'Yes').astype(int)
y_bin = d['y'].to_numpy()
print('default:', d.shape, '  iris:', iris.shape)

## 1. The Naive Bayes Assumption

**LDA** models the full joint class-conditional density f_k(x) as a multivariate Gaussian.
This requires estimating the p×p covariance matrix Σ — when p is large, this is expensive and unreliable.

**Naive Bayes** assumes features are **conditionally independent** given the class:

$$f_k(x) = \prod_{j=1}^{p} f_{kj}(x_j)$$

The joint density factorizes into p marginals. Each marginal $f_{kj}(x_j)$ is estimated separately.

**Posterior** (from Bayes theorem):
$$\Pr(Y=k \mid X=x) \propto \pi_k \cdot \prod_{j=1}^{p} f_{kj}(x_j)$$

In log-space (additive):
$$\log \Pr(Y=k \mid X=x) = \log \pi_k + \sum_{j=1}^{p} \log f_{kj}(x_j) + \text{const}$$

**Parameter count**: p·K marginals vs p(p+1)/2 for LDA's shared Σ. When p is large, NB wins.

The assumption is almost never literally true — features are usually correlated. Yet NB often performs well because:
1. Even a biased boundary can be a good *ranking* of classes
2. Low variance from fewer parameters can outweigh the bias
3. When p >> n, the variance savings are enormous

Despite being "naive", the independence assumption often leads to good classification because classification only requires *ranking* classes by posterior — the absolute probability values can be wrong as long as the ordering is right. This gives NB surprising robustness even when features are clearly correlated.

## 2. Three Choices for the Marginal Density f_kj

| Feature type | Density choice | Parameters per (k, j) |
|---|---|---|
| Continuous numeric | Gaussian: N(μ_kj, σ²_kj) | mean + variance |
| Continuous, non-normal | Kernel density estimate (KDE) | bandwidth h |
| Categorical / ordinal | Multinomial table: P(x_j = l \| Y=k) | counts per level |

**Mixed features** (e.g., Default dataset): use Gaussian for `balance` and `income`, Bernoulli for `student`.
Each feature is modelled independently — no joint distribution needed.

The choice of density family is the key modelling decision in Naive Bayes — it encodes prior knowledge about feature types. Getting this wrong (e.g., using Gaussian for count data) can hurt more than the independence assumption itself. When in doubt: Gaussian for continuous, Multinomial for counts, Bernoulli for binary presence/absence.

In [ ]:
# Visualize class-conditional marginal distributions
X2 = d[['balance', 'income']].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
features = ['balance', 'income']
for col_idx, (feat, ax_row) in enumerate(zip(features, axes)):
    for cls, (label, color, linestyle) in enumerate(
            [('No default', 'C0', '-'), ('Default', 'C3', '--')]):
        data = d[d['y'] == cls][feat]
        ax_row[0].hist(data, bins=40, alpha=0.5, color=color, label=label,
                       density=True)
        from scipy.stats import norm
        mu_est = data.mean(); sigma_est = data.std()
        xs = np.linspace(data.min(), data.max(), 200)
        ax_row[1].plot(xs, norm.pdf(xs, mu_est, sigma_est), color=color,
                       linewidth=2, linestyle=linestyle, label=f'{label}: N({mu_est:.0f}, {sigma_est:.0f}²)')
    ax_row[0].set_xlabel(feat); ax_row[0].set_ylabel('density')
    ax_row[0].set_title(f'Empirical distribution: {feat}')
    ax_row[0].legend(fontsize=9)
    ax_row[1].set_xlabel(feat); ax_row[1].set_ylabel('density')
    ax_row[1].set_title(f'Gaussian NB marginals: {feat}')
    ax_row[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# GaussianNB models each feature as Gaussian — assumes continuous features
gnb = GaussianNB().fit(X2, y_bin)
print('GaussianNB learned parameters:')
print(f'Class priors: {gnb.class_prior_.round(4)}')
print(f'Class means (μ_kj):   {gnb.theta_.round(2)}')
# var_ stores per-class, per-feature variances (diagonal of each class covariance)
print(f'Class variances (σ²): {gnb.var_.round(2)}')
print('\nRows = classes (0=no default, 1=default), Cols = features (balance, income)')

## 3. Toy Example: NB vs LDA Decision Regions

In [ ]:
# Compare NB vs LDA decision boundaries on Default (balance × income)
Xtr, Xte, ytr, yte = train_test_split(X2, y_bin, test_size=0.3, random_state=0, stratify=y_bin)

# GaussianNB models each feature as Gaussian — assumes continuous features
gnb_fit = GaussianNB().fit(Xtr, ytr)
lda_fit = LinearDiscriminantAnalysis().fit(Xtr, ytr)

x0_r = np.linspace(X2[:,0].min()-200, X2[:,0].max()+200, 200)
x1_r = np.linspace(X2[:,1].min()-5000, X2[:,1].max()+5000, 200)
xx, yy = np.meshgrid(x0_r, x1_r)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, title in zip(axes,
        [gnb_fit, lda_fit],
        ['GaussianNB — axis-aligned ellipses', 'LDA — tilted ellipse (shared Σ)']):
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.scatter(X2[y_bin==0, 0], X2[y_bin==0, 1], s=3, alpha=0.3, label='No default')
    ax.scatter(X2[y_bin==1, 0], X2[y_bin==1, 1], s=15, color='C3', alpha=0.7,
               edgecolors='k', linewidths=0.4, label='Default')
    ax.set_xlabel('balance'); ax.set_ylabel('income')
    ax.set_title(title); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# NB assumes P(X1,X2|Y) = P(X1|Y)*P(X2|Y) — false in practice but surprisingly robust
print('GaussianNB AUC:', roc_auc_score(yte, gnb_fit.predict_proba(Xte)[:,1]).round(4))
print('LDA       AUC:', roc_auc_score(yte, lda_fit.predict_proba(Xte)[:,1]).round(4))

In [ ]:
# 2D toy: two bivariate Gaussians with correlated features
rng = np.random.default_rng(42)
# Class 0: correlated features (off-diagonal covariance)
Sigma0 = np.array([[1.0, 0.8], [0.8, 1.0]])
Sigma1 = np.array([[1.0, -0.6], [-0.6, 1.0]])
X0 = rng.multivariate_normal([0, 0], Sigma0, 150)
X1 = rng.multivariate_normal([2, 2], Sigma1, 150)
X_toy = np.vstack([X0, X1])
y_toy = np.array([0]*150 + [1]*150)

gnb_toy = GaussianNB().fit(X_toy, y_toy)
lda_toy = LinearDiscriminantAnalysis().fit(X_toy, y_toy)
qda_toy = QuadraticDiscriminantAnalysis().fit(X_toy, y_toy)

xx_t, yy_t = np.meshgrid(np.linspace(-4, 6, 200), np.linspace(-4, 6, 200))
g_t = np.c_[xx_t.ravel(), yy_t.ravel()]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, model, title in zip(axes,
        [gnb_toy, lda_toy, qda_toy],
        ['Naive Bayes (axis-aligned)', 'LDA (shared Σ, linear)', 'QDA (per-class Σ, quadratic)']):
    Z = model.predict(g_t).reshape(xx_t.shape)
    ax.contourf(xx_t, yy_t, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(X0[:,0], X0[:,1], s=10, alpha=0.6, label='Class 0')
    ax.scatter(X1[:,0], X1[:,1], s=10, alpha=0.6, color='C3', label='Class 1')
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8)
    auc_val = roc_auc_score(y_toy, model.predict_proba(X_toy)[:,1])
    ax.set_xlabel(f'AUC = {auc_val:.3f}')
plt.tight_layout(); plt.show()
print('QDA fits best here (different Σ per class). NB misses the correlations.')

## 4. Connections

### NB ↔ LDA / QDA

- **NB = QDA with diagonal Σ_k**: QDA with the off-diagonal covariance terms forced to zero.
- **NB = LDA** when additionally all classes share the same diagonal (same σ²_j for all k).

NB and LDA are equivalent when features are Gaussian and independent — but NB estimates each feature's variance separately (per class), rather than pooling across classes as LDA does. The pooling in LDA can help when n is small; NB's per-class variances are more flexible.

### NB ↔ GAMs (Generalized Additive Models)

The log-posterior ratio between class k and class l:

$$\log\frac{\Pr(Y=k|X=x)}{\Pr(Y=l|X=x)} = \log\frac{\pi_k}{\pi_l} + \sum_{j=1}^{p} \underbrace{\log\frac{f_{kj}(x_j)}{f_{lj}(x_j)}}_{g_j(x_j)}$$

This is **additive in functions of individual features** $g_j(x_j)$ — exactly the structure of a GAM.
With Gaussian marginals, $g_j(x_j)$ is quadratic → the decision boundary is a quadratic surface.
With KDE marginals, $g_j(x_j)$ is non-parametric → a flexible non-linear boundary.

In [ ]:
# Illustrate: NB log-posterior is additive — each feature contributes independently
# Compute log-posterior contribution of balance and income separately
from scipy.stats import norm as sp_norm

x_bal = np.linspace(0, 2700, 300)
x_inc = np.linspace(10000, 70000, 300)

# Learned parameters from GaussianNB
mu_bal = gnb_fit.theta_[:, 0]    # [mu_0_balance, mu_1_balance]
sig_bal = np.sqrt(gnb_fit.var_[:, 0])
mu_inc = gnb_fit.theta_[:, 1]
sig_inc = np.sqrt(gnb_fit.var_[:, 1])

# g_j(x_j) = log(f_1j(x_j) / f_0j(x_j)) for each feature
g_bal = sp_norm.logpdf(x_bal, mu_bal[1], sig_bal[1]) - sp_norm.logpdf(x_bal, mu_bal[0], sig_bal[0])
g_inc = sp_norm.logpdf(x_inc, mu_inc[1], sig_inc[1]) - sp_norm.logpdf(x_inc, mu_inc[0], sig_inc[0])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_bal, g_bal, linewidth=2)
axes[0].axhline(0, color='k', linestyle='--', alpha=0.4)
axes[0].set_xlabel('balance'); axes[0].set_ylabel('g_balance(x)')
axes[0].set_title('GAM component: balance → log likelihood ratio')
axes[1].plot(x_inc, g_inc, linewidth=2, color='C1')
axes[1].axhline(0, color='k', linestyle='--', alpha=0.4)
axes[1].set_xlabel('income'); axes[1].set_ylabel('g_income(x)')
axes[1].set_title('GAM component: income → log likelihood ratio')
plt.tight_layout(); plt.show()
print('NB log-posterior = log(π₁/π₀) + g_balance(balance) + g_income(income)')
print('Each component is quadratic (Gaussian marginals) — additive GAM structure.')

## 5. sklearn Naive Bayes Variants

sklearn offers four NB variants — the right one depends on feature type:
- **GaussianNB**: continuous features (sensor readings, measurements, financial data)
- **MultinomialNB**: non-negative integer counts (word frequencies, n-gram counts) — the standard choice for text
- **BernoulliNB**: binary features (word presence/absence) — better than Multinomial when document length varies a lot
- **CategoricalNB**: nominal categorical features (e.g., colour, region)

For mixed feature types, fit separate NB models per feature group and combine log-posteriors.

In [ ]:
# MultinomialNB models feature counts — suited for text (word frequencies)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Simulated email data
emails = [
    'free money click here now', 'win prize free lottery',
    'buy cheap pills online now', 'make money fast free offer',
    'click free offer win cash', 'meeting tomorrow agenda attached',
    'project update please review document', 'lunch tomorrow at noon',
    'quarterly report attached review', 'team sync at three pm today',
    'free offer win now click', 'cheap discount buy now pills',
    'report due friday please review', 'meeting minutes attached pdf',
]
labels = [1,1,1,1,1,0,0,0,0,0,1,1,0,0]  # 1=spam, 0=ham

pipe = Pipeline([
    ('vect', CountVectorizer()),
    ('nb',   MultinomialNB(alpha=1.0))  # Laplace smoothing
])
pipe.fit(emails, labels)

test_emails = ['free money win now', 'meeting agenda tomorrow']
preds = pipe.predict(test_emails)
# log-space computation avoids numerical underflow when multiplying many small probabilities
probas = pipe.predict_proba(test_emails)
print('MultinomialNB text classification:')
for email, pred, proba in zip(test_emails, preds, probas):
    print(f'  "{email}"')
    print(f'    → {"SPAM" if pred==1 else "HAM"}, P(spam) = {proba[1]:.4f}')

In [ ]:
# BernoulliNB models binary features — suited for binary word presence
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB

# Binary: is each word present? (not how many times)
vect_bin = CountVectorizer(binary=True)
X_bin_text = vect_bin.fit_transform(emails)
bnb = BernoulliNB().fit(X_bin_text, labels)

X_test_bin = vect_bin.transform(test_emails)
preds_b = bnb.predict(X_test_bin)
print('BernoulliNB (word presence, not count):')
for email, pred in zip(test_emails, preds_b):
    print(f'  "{email}" → {"SPAM" if pred==1 else "HAM"}')

# Most spammy words
vocab = np.array(vect_bin.get_feature_names_out())
log_ratio = bnb.feature_log_prob_[1] - bnb.feature_log_prob_[0]
top5 = vocab[np.argsort(log_ratio)[-5:][::-1]]
print(f'\nTop 5 spam-indicating words: {top5}')

## 6. Benchmark: NB vs LDA vs QDA vs LR

NB often wins on **small n** (few parameters to estimate, so low variance dominates) but tends to lose on **large n with correlated features** where richer models like LR or LDA can exploit covariance structure. Think of it as the high-bias, low-variance end of the classifier spectrum.

In [ ]:
# Benchmark all four classifiers on Default dataset
from sklearn.model_selection import cross_val_score

X_def = d[['balance', 'income']].to_numpy()
models = [
    ('LR',  LogisticRegression(max_iter=2000)),
    ('LDA', LinearDiscriminantAnalysis()),
    ('QDA', QuadraticDiscriminantAnalysis()),
    ('NB',  GaussianNB()),
]
print('5-fold CV AUC on Default (balance + income):')
results = {}
for name, model in models:
    scores = cross_val_score(model, X_def, y_bin, cv=5, scoring='roc_auc')
    results[name] = scores
    print(f'  {name:5s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Benchmark on Iris (3-class, accuracy)
le = LabelEncoder().fit(iris['species'])
y_iris = le.transform(iris['species'])
X_iris = iris[['sepal_length','sepal_width','petal_length','petal_width']].to_numpy()

print('5-fold CV Accuracy on Iris (4 features, 3 classes):')
for name, model in models:
    scores = cross_val_score(model, X_iris, y_iris, cv=5)
    print(f'  {name:5s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Demonstrate NB advantage: high-dimensional setting (p >> n)
from sklearn.datasets import make_classification

rng2 = np.random.default_rng(0)
print('AUC in high-dimensional setting (p=500 features, n=200 samples):')
for trial in range(3):
    X_hi, y_hi = make_classification(n_samples=200, n_features=500,
                                      n_informative=20, n_redundant=0,
                                      random_state=trial)
    Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_hi, y_hi, test_size=0.3,
                                                    random_state=trial)
    results_hi = {}
    for name, model in [('NB', GaussianNB()),
                         ('LR', LogisticRegression(max_iter=500))]:
        m = model.fit(Xh_tr, yh_tr)
        # log-space computation avoids numerical underflow when multiplying many small probabilities
        results_hi[name] = roc_auc_score(yh_te, m.predict_proba(Xh_te)[:,1])
    print(f'  Trial {trial}: NB = {results_hi["NB"]:.3f}, LR = {results_hi["LR"]:.3f}')
print('NB is competitive with LR even at p=500 — no need to estimate a covariance matrix.')

## Recap

1. **Conditional independence**: f_k(x) = ∏_j f_kj(x_j). One marginal per feature per class.
2. **Posterior**: Pr(Y=k|X) ∝ π_k · ∏_j f_kj(x_j). Additive in log-space.
3. **Three density choices**: Gaussian (numeric), KDE (non-normal), multinomial (categorical).
4. **Mixed features**: Gaussian for numeric + Bernoulli for binary — handled naturally.
5. **NB ↔ GAMs**: log-posterior ratio = sum of feature-level log-likelihood ratios — a GAM.
6. **NB = QDA with diagonal Σ_k** = LDA when additionally same σ²_j across classes.
7. **NB wins**: p >> n (text, genomics). **NB loses**: strong within-class feature correlations.

---

# Exercises

## Exercise 1

**Task 1.** Compare GaussianNB vs LDA on the Default dataset with all three predictors
(`balance`, `income`, `student`). Encode `student` as 0/1. Report AUC for both.

In [ ]:
# your code here

### Exercise 1 — Solution

In [ ]:
d['student_num'] = (d['student'] == 'Yes').astype(float)
X3 = d[['balance', 'income', 'student_num']].to_numpy()
Xtr3, Xte3, ytr3, yte3 = train_test_split(X3, y_bin, test_size=0.3, random_state=0, stratify=y_bin)
for name, model in [('GaussianNB', GaussianNB()), ('LDA', LinearDiscriminantAnalysis())]:
    model.fit(Xtr3, ytr3)
    auc_val = roc_auc_score(yte3, model.predict_proba(Xte3)[:,1])
    print(f'{name}: AUC = {auc_val:.4f}')

## Exercise 2

**Task 2.** Build a spam classifier using `MultinomialNB` on the 20 Newsgroups dataset
(binary: `sci.med` vs `rec.sport.hockey`). Report accuracy and AUC.

In [ ]:
# your code here

### Exercise 2 — Solution

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

cats = ['sci.med', 'rec.sport.hockey']
train_data = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'))

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, sublinear_tf=True)),
    ('nb',    MultinomialNB())
])
pipe.fit(train_data.data, train_data.target)
proba = pipe.predict_proba(test_data.data)[:, 1]
print(f'Accuracy: {pipe.score(test_data.data, test_data.target):.4f}')
print(f'AUC:      {roc_auc_score(test_data.target, proba):.4f}')

## Exercise 3

**Task 3.** Implement Naive Bayes from scratch for the binary Default dataset
(two Gaussian marginals). Compute the posterior probability manually and compare to `GaussianNB`.

In [ ]:
# your code here

### Exercise 3 — Solution

In [ ]:
from scipy.stats import norm

# Fit parameters by hand
pi = {k: (y_bin == k).mean() for k in [0, 1]}
mu = {k: X2[y_bin == k].mean(axis=0) for k in [0, 1]}
sigma = {k: X2[y_bin == k].std(axis=0) for k in [0, 1]}

def nb_posterior(x, pi, mu, sigma):
    log_posts = []
    for k in [0, 1]:
        log_p = np.log(pi[k]) + sum(
            norm.logpdf(x[j], mu[k][j], sigma[k][j]) for j in range(len(x))
        )
        log_posts.append(log_p)
    log_posts = np.array(log_posts)
    log_posts -= log_posts.max()  # numerical stability
    probs = np.exp(log_posts); probs /= probs.sum()
    return probs

# Compare on first 5 test points
# var_smoothing adds a small variance floor — prevents zero-probability issues
gnb_ref = GaussianNB(var_smoothing=1e-9).fit(X2, y_bin)
print('Manual NB vs sklearn GaussianNB (P(default=1)):') 
for i in range(5):
    manual = nb_posterior(Xte[i], pi, mu, sigma)[1]
    sklearn_val = gnb_ref.predict_proba(Xte[i:i+1])[0, 1]
    print(f'  x={Xte[i].round(0)}: manual={manual:.6f}, sklearn={sklearn_val:.6f}')

## Exercise 4

**Task 4 — Feature independence check.**
On the Default dataset (`balance`, `income`, `student_binary`), compute the pairwise mutual information between each pair of features using `sklearn.metrics.mutual_info_score`. Discretize continuous features into 10 equal-width bins with `pd.cut`. Print a 3×3 MI matrix. Then compare GaussianNB vs LDA accuracy — does higher feature correlation hurt NB more?

In [ ]:
# your code here

### Exercise 4 — Solution

In [ ]:
from sklearn.metrics import mutual_info_score

d['student_binary'] = (d['student'] == 'Yes').astype(float)
features_mi = ['balance', 'income', 'student_binary']

# Discretize continuous features into 10 equal-width bins for MI computation
disc = {}
for feat in features_mi:
    if feat == 'student_binary':
        disc[feat] = d[feat].astype(int)
    else:
        disc[feat] = pd.cut(d[feat], bins=10, labels=False)

# Build 3x3 MI matrix
mi_matrix = pd.DataFrame(index=features_mi, columns=features_mi, dtype=float)
for fi in features_mi:
    for fj in features_mi:
        mi_matrix.loc[fi, fj] = mutual_info_score(disc[fi], disc[fj])

print('Pairwise Mutual Information matrix:')
print(mi_matrix.round(4))

# Compare GaussianNB vs LDA with all three features
X3_mi = d[features_mi].to_numpy()
Xtr3m, Xte3m, ytr3m, yte3m = train_test_split(X3_mi, y_bin, test_size=0.3, random_state=0, stratify=y_bin)
print('\nAUC comparison (3 features: balance, income, student_binary):')
for name, model in [('GaussianNB', GaussianNB()), ('LDA', LinearDiscriminantAnalysis())]:
    model.fit(Xtr3m, ytr3m)
    auc_val = roc_auc_score(yte3m, model.predict_proba(Xte3m)[:, 1])
    print(f'  {name}: AUC = {auc_val:.4f}')
print('\nBalance and income likely show low MI (independent); student is binary so MI is lower.')
print('NB and LDA perform similarly because the key predictor (balance) has low correlation with others.')

## Exercise 5

**Task 5 — Text classification toy example.**
Create a small dataset of 12 documents (6 "sports", 6 "politics") with word counts for 5 words: `['goal', 'player', 'vote', 'law', 'election']`. Use `MultinomialNB` to classify 3 new documents. Print the predicted class and probability for each. Then try `BernoulliNB` (convert counts to binary presence) and compare the predictions.

In [ ]:
# your code here

### Exercise 5 — Solution

In [ ]:
# Words: ['goal', 'player', 'vote', 'law', 'election']
# 12 documents: 6 sports (label 0), 6 politics (label 1)
X_docs = np.array([
    # sports documents (goal, player counts high)
    [4, 3, 0, 0, 0],
    [3, 5, 0, 1, 0],
    [5, 2, 0, 0, 0],
    [2, 4, 1, 0, 0],
    [6, 3, 0, 0, 0],
    [3, 3, 0, 0, 1],
    # politics documents (vote, law, election counts high)
    [0, 0, 4, 3, 2],
    [1, 0, 3, 4, 3],
    [0, 1, 5, 2, 4],
    [0, 0, 2, 5, 3],
    [0, 1, 4, 3, 5],
    [1, 0, 3, 4, 2],
])
y_docs = np.array([0]*6 + [1]*6)  # 0=sports, 1=politics
class_names = ['sports', 'politics']
word_names  = ['goal', 'player', 'vote', 'law', 'election']

# 3 new documents to classify
X_new = np.array([
    [3, 2, 0, 0, 0],   # clearly sports
    [0, 0, 3, 2, 4],   # clearly politics
    [2, 1, 2, 1, 1],   # ambiguous
])

# MultinomialNB models feature counts — suited for text (word frequencies)
mnb = MultinomialNB(alpha=1.0).fit(X_docs, y_docs)
print('MultinomialNB predictions:')
for i, (doc, pred, prob) in enumerate(zip(X_new, mnb.predict(X_new), mnb.predict_proba(X_new))):
    print(f'  doc {i+1} {dict(zip(word_names, doc))}: {class_names[pred]} '
          f'(P(sports)={prob[0]:.3f}, P(politics)={prob[1]:.3f})')

# BernoulliNB models binary features — suited for binary word presence
X_new_bin = (X_new > 0).astype(int)
X_docs_bin = (X_docs > 0).astype(int)
# BernoulliNB models binary features — suited for binary word presence
bnb2 = BernoulliNB(alpha=1.0).fit(X_docs_bin, y_docs)
print('\nBernoulliNB predictions (binary presence):')
for i, (doc, pred, prob) in enumerate(zip(X_new_bin, bnb2.predict(X_new_bin), bnb2.predict_proba(X_new_bin))):
    print(f'  doc {i+1} presence={dict(zip(word_names, doc))}: {class_names[pred]} '
          f'(P(sports)={prob[0]:.3f}, P(politics)={prob[1]:.3f})')
print('\nNote: Multinomial uses exact counts; Bernoulli only sees presence. Ambiguous doc may differ.')

## Exercise 6

**Task 6 — Calibration.**
GaussianNB posteriors are often overconfident (predicted probabilities near 0 or 1). On the Default dataset, plot a calibration curve for `GaussianNB` vs `LogisticRegression` using `sklearn.calibration.calibration_curve` with `n_bins=10`. Then apply `CalibratedClassifierCV(gnb, method='isotonic', cv=3)` and re-plot. Does isotonic calibration fix the overconfidence?

In [ ]:
# your code here

### Exercise 6 — Solution

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Use 3 features for a more realistic calibration test
d['student_binary'] = (d['student'] == 'Yes').astype(float)
X_cal = d[['balance', 'income', 'student_binary']].to_numpy()
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(X_cal, y_bin, test_size=0.3,
                                                random_state=0, stratify=y_bin)

# Fit the three models
gnb_cal = GaussianNB().fit(Xtr_c, ytr_c)
lr_cal  = LogisticRegression(max_iter=2000).fit(Xtr_c, ytr_c)
# CalibratedClassifierCV wraps any classifier and post-processes its probabilities
cal_gnb = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3).fit(Xtr_c, ytr_c)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')

for model, label, color in [
        (gnb_cal,  'GaussianNB (raw)',        'C3'),
        (lr_cal,   'LogisticRegression',       'C0'),
        (cal_gnb,  'GaussianNB (calibrated)',  'C2'),
]:
    prob_pos = model.predict_proba(Xte_c)[:, 1]
    # log-space computation avoids numerical underflow when multiplying many small probabilities
    frac_pos, mean_pred = calibration_curve(yte_c, prob_pos, n_bins=10)
    ax.plot(mean_pred, frac_pos, 's-', label=label, color=color)

ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration curves on Default dataset')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print('Raw GaussianNB is often overconfident — probabilities cluster near 0 and 1.')
print('Isotonic calibration should bring the NB curve closer to the diagonal.')
print('Logistic Regression is generally well-calibrated by construction.')